In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsproyectoyr1";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_pry CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_pry
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_pry.raw;
DROP SCHEMA IF EXISTS catalog_pry.bronze;
DROP SCHEMA IF EXISTS catalog_pry.silver;
DROP SCHEMA IF EXISTS catalog_pry.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_pry.raw;
CREATE SCHEMA IF NOT EXISTS catalog_pry.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_pry.silver;
CREATE SCHEMA IF NOT EXISTS catalog_pry.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.bronze.categories (
CategoryID integer,
CategoryName string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/categories"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.bronze.cities (
CityID integer,
CityName string,
Zipcode integer,
CountryID integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/cities"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.bronze.countries (
CountryID integer,
CountryName string,
CountryCode string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/countries"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.bronze.customers (
CustomerID integer,
FirstName string,
MiddleInitial string,
LastName string,
CityID integer,
Address string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/customers"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.bronze.employees (
EmployeeID integer,
FirstName string,
MiddleInitial string,
LastName string,
BirthDate date,
Gender string,
CityID integer,
HireDate timestamp,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/employees"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.bronze.products (
    ProductID integer,
    ProductName string,
    Price decimal,
    CategoryID integer,
    Class string,
    ModifyDate timestamp,
    Resistant string,
    IsAllergic string,
    VitalityDays integer,
    ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/products";



In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.bronze.sales (
    SalesID integer,
    SalesPersonID integer,
    CustomerID integer,
    ProductID integer,
    Quantity integer,
    Discount decimal(10,2),
    TotalPrice decimal(10,2),
    SalesDate timestamp,
    TransactionNumber string,
    ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/sales";

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.silver.orders_enriched (
    SalesID integer,
    CustomerID integer,
    CustomerName string,
    CustomerEmail string,
    ProductID integer,
    ProductName string,
    Category string,
    Quantity integer,
    Price double,
    TotalValue double,
    SalesDate timestamp,
    ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/orders_enriched";

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_pry.golden.sales_summary (
    Category string,
    CustomerName string,
    TotalSales decimal(12,2),
    ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/sales_summary";